In [8]:
import sys
# !{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install seaborn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3.13 -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [ ]:
class SalesDataAnalyzer:
    
    def _init_(self, file_path=None):
        self.data = None
        self.last_plot = None
        if file_path:
            self.load_data(file_path)
    
    def __del__(self):
        print("SalesDataAnlyzer object deleted, cleanup done.")
        
    def load_data(self, file_path):
        try:
            self.data = pd.read_csv(file_path)
        except FileExistsError:
            print("File not Found:", file_path)
            return
        except Exception as e:
            print("File not Found: ", file_path)
            return
        
        print("Datset loaded successfully!", len(self.data), "rows")

        if "category" in self.data.columns:
            self.data["main_category"] = self.data["category"].str.split("|").str[0]
        
        self.clean_column()
        
    def get_data(self):
        return self.data
    
    def clean_columns(self):
        # discounted,actual_price have rupee and commas,
        for col in ["discounted_price", "actual_price"]:
            if col in self.data.columns:
                self.data[col] = (
                    self.data[col]
                    .astype(str)
                    .str.replace("₹", "", regex=False)
                    .str.replace(",", "", regex=False)
                )
                self.data[col] = pd.to_numeric(self.data[col], errors="coerce")
 
        # discount has a % sign, e.g. "64%"
        if "discount_percentage" in self.data.columns:
            self.data["discount_percentage"] = (
                self.data["discount_percentage"]
                .astype(str)
                .str.replace("%", "", regex=False)
            )
            self.data["discount_percentage"] = pd.to_numeric(
                self.data["discount_percentage"], errors="coerce"
            )
 
        if "rating_count" in self.data.columns:
            self.data["rating_count"] = (
                self.data["rating_count"].astype(str).str.replace(",", "", regex=False)
            )
            self.data["rating_count"] = pd.to_numeric(
                self.data["rating_count"], errors="coerce"
            )
            
        if "rating" in self.data.columns:
            self.data["rating"] = pd.to_numeric(self.data["rating"], errors="coerce")
 
        print("Cleaned price, discount, rating and rating_count columns.")
    
    def explore_data(self):
        if not self.has_data():
            return
        print("1. First 5 rows")
        print("2. Last 5 rows")
        print("3. Column names")
        print("4. Data types")
        print("5. Basic info / describe")
        choice = input("Enter your choice: ")
 
        if choice == "1":
            print(self.data.head())
        elif choice == "2":
            print(self.data.tail())
        elif choice == "3":
            print(list(self.data.columns))
        elif choice == "4":
            print(self.data.dtypes)
        elif choice == "5":
            print(self.data.info())
            print(self.data.describe())
        else:
            print("Invalid choice")
            
    def clean_data(self):
        #this function handle nan Values
        if not self.has_data():
            return
        print("1. Show rows with missing values")
        print("2. Fill missing numeric values with mean")
        print("3. Drop rows with missing values")
        print("4. Replace missing values with a value you type in")
        choice = input("Enter your choice: ")
 
        if choice == "1":
            missing_rows = self.data[self.data.isnull()]
            if len(missing_rows) == 0:
                print("No missing values found in the dataset!")
            else:
                print(missing_rows)
        elif choice == "2":
            numeric_cols = self.data.select_dtypes(include="number").columns
            for col in numeric_cols:
                self.data[col] = self.data[col].fillna(self.data[col].mean())
            print("Missing numeric values filled with column mean.")
        elif choice == "3":
            before = len(self.data)
            self.data = self.data.dropna()
            print("Dropped", before - len(self.data), "rows.")
        elif choice == "4":
            value = input("Enter the value to fill with: ")
            self.data = self.data.fillna(value)
            print("Missing values replaced with", value)
        else:
            print("Invalid choice") 
            
    def math_operations(self):
        if not self.has_data():
            return
        
        self.data["savings_rupees"] = self.data["actual_price"] - self.data["discounted_price"]
            
        self.data["price_per_rating"] = self.data["discounted_price"] / self.data["rating"]
        print("Added 'savings_rupees' and 'price_per_rating' columns.")
        
    def search_sort_filter(self):
        if not self.has_data():
            return
        print("1. Search")
        print("2. Sort")
        print("3. Filter")
        choice = input("Enter your choice: ")
 
        if choice == "1":
            column = input("Column to search in: ")
            value = input("Value to search for: ")
            result = self.data[self.data[column].astype(str).str.contains(value, case=False, na=False)]
            print(result if len(result) > 0 else "No matching records found.")
        elif choice == "2":
            column = input("Column to sort by: ")
            order = input("Ascending? (y/n): ")
            print(self.data.sort_values(by=column, ascending=(order.lower() != "n")))
        elif choice == "3":
            column = input("Column to filter on: ")
            operator = input("Operator (>, <, ==): ")
            value = float(input("Value: "))
            if operator == ">":
                print(self.data[self.data[column] > value])
            elif operator == "<":
                print(self.data[self.data[column] < value])
            else:
                print(self.data[self.data[column] == value])
        else:
            print("Invalid choice")
 
    def aggregate_functions(self):
        # requirement: sum, mean, count etc grouped by a column
        if not self.has_data():
            return
        value_col = input("Numeric column (e.g. discounted_price): ") or "discounted_price"
        group_col = input("Group by column (e.g. main_category): ") or "main_category"
        result = self.data.groupby(group_col)[value_col].agg(["sum", "mean", "count"])
        print(result)
  
    def pivot_table(self):
        if not self.has_data():
            return
        values = input("Values column (e.g. discounted_price): ") or "discounted_price"
        index = input("Index column (e.g. main_category): ") or "main_category"
        table = pd.pivot_table(self.data, values=values, index=index, aggfunc="mean")
        print(table)
 
    def reindex_relabel(self):
        if not self.has_data():
            return
        self.data = self.data.reset_index(drop=True)
        self.data = self.data.rename(columns={"discounted_price": "current_price"})
        print("Reset the row index and renamed 'discounted_price' to 'current_price'.")    

    def statistical_analysis(self):
        # requirement: std, var, percentiles, describe
        if not self.has_data():
            return
        column = input("Column to analyze (e.g. rating): ") or "rating"
        data = self.data[column].dropna()
        print(data.describe())
        print("std:", data.std())
        print("var:", data.var())
        print("25th percentile:", data.quantile(0.25))
        print("50th percentile:", data.quantile(0.50))
        print("75th percentile:", data.quantile(0.75))
        
    #------MatplotLib------------
        def visualize_data(self):
        if not self.has_data():
            return
        print("1. Bar Plot")
        print("2. Line Plot")
        print("3. Scatter Plot")
        print("4. Pie Chart")
        print("5. Box Plot")
        print("6. Histogram")
        print("7. Violin Plot")
        print("8. Stack Plot")
        print("9. Step Chart")
        choice = input("Enter your choice: ")
 
        fig, ax = plt.subplots(figsize=(8, 5))
 
        if choice == "1":
            self.data.groupby("main_category")["rating_count"].sum().plot(kind="bar", ax=ax)
            ax.set_title("Total Rating Count by Category")
 
        elif choice == "2":
            x_col = input("X-axis column: ")
            y_col = input("Y-axis column: ")
            ax.plot(self.data[x_col], self.data[y_col])
            ax.set_title(y_col + " over " + x_col)
 
        elif choice == "3":
            x_col = input("X-axis column: ")
            y_col = input("Y-axis column: ")
            print("Generating scatter plot...")
            ax.scatter(self.data[x_col], self.data[y_col])
            ax.set_title(y_col + " vs " + x_col)
            print("Scatter plot displayed successfully!")
 
        elif choice == "4":
            self.data["main_category"].value_counts().plot(kind="pie", ax=ax, autopct="%1.1f%%")
            ax.set_ylabel("")
            ax.set_title("Product Share by Category")
 
        elif choice == "5":
            self.data.boxplot(column="discounted_price", by="main_category", ax=ax, rot=45)
            ax.set_title("Price Distribution by Category")
 
        elif choice == "6":
            ax.hist(self.data["rating"].dropna(), bins=15)
            ax.set_title("Rating Distribution")
 
        elif choice == "7":
            ax.violinplot(self.data["rating"].dropna())
            ax.set_title("Rating Distribution (Violin)")
 
        elif choice == "8":
            pivot = self.data.pivot_table(values="rating_count", index="main_category", aggfunc="sum")
            ax.stackplot(range(len(pivot)), pivot["rating_count"])
            ax.set_title("Rating Count Stack Plot")
 
        elif choice == "9":
            grouped = self.data.groupby("main_category")["discount_percentage"].mean()
            ax.step(grouped.index, grouped.values)
            ax.set_title("Average Discount % by Category")
            ax.tick_params(axis="x", rotation=45)
 
        else:
            print("Invalid choice")
            plt.close(fig)
            return
 
        fig.tight_layout()
        self.last_plot = fig
        plt.show()
    #-------Matplotlib
    def visualize_data(self):
        if not self.has_data():
            return
        print("1. Bar Plot")
        print("2. Line Plot")
        print("3. Scatter Plot")
        print("4. Pie Chart")
        print("5. Box Plot")
        print("6. Histogram")
        print("7. Violin Plot")
        print("8. Stack Plot")
        print("9. Step Chart")
        choice = input("Enter your choice: ")
 
        fig, ax = plt.subplots(figsize=(8, 5))
 
        if choice == "1":
            self.data.groupby("main_category")["rating_count"].sum().plot(kind="bar", ax=ax)
            ax.set_title("Total Rating Count by Category")
 
        elif choice == "2":
            x_col = input("X-axis column: ")
            y_col = input("Y-axis column: ")
            ax.plot(self.data[x_col], self.data[y_col])
            ax.set_title(y_col + " over " + x_col)
 
        elif choice == "3":
            x_col = input("X-axis column: ")
            y_col = input("Y-axis column: ")
            print("Generating scatter plot...")
            ax.scatter(self.data[x_col], self.data[y_col])
            ax.set_title(y_col + " vs " + x_col)
            print("Scatter plot displayed successfully!")
 
        elif choice == "4":
            self.data["main_category"].value_counts().plot(kind="pie", ax=ax, autopct="%1.1f%%")
            ax.set_ylabel("")
            ax.set_title("Product Share by Category")
 
        elif choice == "5":
            self.data.boxplot(column="discounted_price", by="main_category", ax=ax, rot=45)
            ax.set_title("Price Distribution by Category")
 
        elif choice == "6":
            ax.hist(self.data["rating"].dropna(), bins=15)
            ax.set_title("Rating Distribution")
 
        elif choice == "7":
            ax.violinplot(self.data["rating"].dropna())
            ax.set_title("Rating Distribution (Violin)")
 
        elif choice == "8":
            pivot = self.data.pivot_table(values="rating_count", index="main_category", aggfunc="sum")
            ax.stackplot(range(len(pivot)), pivot["rating_count"])
            ax.set_title("Rating Count Stack Plot")
 
        elif choice == "9":
            grouped = self.data.groupby("main_category")["discount_percentage"].mean()
            ax.step(grouped.index, grouped.values)
            ax.set_title("Average Discount % by Category")
            ax.tick_params(axis="x", rotation=45)
 
        else:
            print("Invalid choice")
            plt.close(fig)
            return
 
        fig.tight_layout()
        self.last_plot = fig
        plt.show()
 
    def save_visualization(self):
        if self.last_plot is None:
            print("No visualization to save yet. Generate one first (option 6).")
            return
        filename = input("Enter file name to save the plot (e.g., scatter_plot.png): ")
        self.last_plot.savefig("output/" + filename)
        print("Visualization saved as", filename, "successfully!")
                      
    # ---------------- seaborn visualization ----------------
 
    def visualize_seaborn(self):
        if not self.has_data():
            return
        print("1. Heatmap (correlation)")
        print("2. Pair Plot")
        print("3. Box Plot")
        print("4. Violin Plot")
        choice = input("Enter your choice: ")
 
        numeric_cols = self.data.select_dtypes(include="number")
 
        if choice == "1":
            fig, ax = plt.subplots(figsize=(7, 5))
            sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", ax=ax)
            ax.set_title("Correlation Heatmap")
            self.last_plot = fig
 
        elif choice == "2":
            g = sns.pairplot(numeric_cols.dropna())
            self.last_plot = g.fig
 
        elif choice == "3":
            fig, ax = plt.subplots(figsize=(8, 5))
            sns.boxplot(data=self.data, x="main_category", y="discounted_price", ax=ax)
            ax.tick_params(axis="x", rotation=45)
            ax.set_title("Price by Category")
            self.last_plot = fig
 
        elif choice == "4":
            fig, ax = plt.subplots(figsize=(8, 5))
            sns.violinplot(data=self.data, x="main_category", y="rating", ax=ax)
            ax.tick_params(axis="x", rotation=45)
            ax.set_title("Rating by Category")
            self.last_plot = fig
 
        else:
            print("Invalid choice")
            return
 
        plt.show()          

In [ ]:
dataSet1 = SalesDataAnalyzer()
dataSet1.load_data("/Users/Project/pythonSelf/pythonSelf/RevisedImp/AssignMentProjects/amazon.csv")

Datset loaded successfully! 1465 rows


In [ ]:
dataSet1.get_data()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link,main_category
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,Computers&Accessories
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...,Computers&Accessories
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...,Computers&Accessories
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...,Computers&Accessories
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...,Computers&Accessories
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1460,B08L7J3T31,Noir Aqua - 5pcs PP Spun Filter + 1 Spanner | ...,Home&Kitchen|Kitchen&HomeAppliances|WaterPurif...,₹379,₹919,59%,4,"1,090",SUPREME QUALITY 90 GRAM 3 LAYER THIK PP SPUN F...,"AHITFY6AHALOFOHOZEOC6XBP4FEA,AFRABBODZJZQB6Z4U...","Prabha ds,Raghuram bk,Real Deal,Amazon Custome...","R3G3XFHPBFF0E8,R3C0BZCD32EIGW,R2EBVBCN9QPD9R,R...","Received the product without spanner,Excellent...","I received product without spanner,Excellent p...",https://m.media-amazon.com/images/I/41fDdRtjfx...,https://www.amazon.in/Noir-Aqua-Spanner-Purifi...,Home&Kitchen
1461,B01M6453MB,Prestige Delight PRWO Electric Rice Cooker (1 ...,Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...,"₹2,280","₹3,045",25%,4.1,"4,118","230 Volts, 400 watts, 1 Year","AFG5FM3NEMOL6BNFRV2NK5FNJCHQ,AGEINTRN6Z563RMLH...","Manu Bhai,Naveenpittu,Evatira Sangma,JAGANNADH...","R3DD

In [ ]:
def dataframe_operations(analyzer):
    print("""
-- DataFrame Operations --
1. NumPy array creation, indexing, slicing
2. Mathematical operations (savings, price per rating)
3. Combine with another CSV file
4. Split data by a column
5. Search / Sort / Filter
6. Aggregate functions (sum/mean/count)
7. Pivot table
8. Reindex and relabel columns
9. Groupby + transform
""")
    choice = input("Enter your choice: ")
 
    if choice == "1":
        analyzer.array_demo()
    elif choice == "2":
        analyzer.math_operations()
    elif choice == "3":
        analyzer.combine_data()
    elif choice == "4":
        analyzer.split_data()
    elif choice == "5":
        analyzer.search_sort_filter()
    elif choice == "6":
        analyzer.aggregate_functions()
    elif choice == "7":
        analyzer.pivot_table()
    elif choice == "8":
        analyzer.reindex_relabel()
    elif choice == "9":
        analyzer.groupby_transform()
    else:
        print("Invalid choice")
 
 
def main():
    analyzer = SalesDataAnalyzer()
 
    while True:
        print("""
========== Data Analysis & Visualization Program ==========
1. Load Dataset
2. Explore Data
3. Perform DataFrame Operations
4. Handle Missing Data
5. Generate Descriptive Statistics
6. Data Visualization
7. Save Visualization
8. Exit
=============================================================
""")
        choice = input("Enter your choice: ")
 
        if choice == "1":
            path = input("Enter the path of the dataset (CSV file): ")
            analyzer.load_data(path)
 
        elif choice == "2":
            analyzer.explore_data()
 
        elif choice == "3":
            dataframe_operations(analyzer)
 
        elif choice == "4":
            analyzer.clean_data()
 
        elif choice == "5":
            analyzer.statistical_analysis()
 
        elif choice == "6":
            print("1. Matplotlib")
            print("2. Seaborn")
            sub_choice = input("Enter your choice: ")
            if sub_choice == "1":
                analyzer.visualize_data()
            elif sub_choice == "2":
                analyzer.visualize_seaborn()
            else:
                print("Invalid choice")
 
        elif choice == "7":
            analyzer.save_visualization()
 
        elif choice == "8":
            print("Exiting the program. Goodbye!")
            break
 
        else:
            print("Invalid choice, please try again.")
            
main()